In [ ]:
# Imports
import os
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import random

In [ ]:
# chossing the device
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
# defining the neural network
class NeuralNetwork(nn.Module):
    def __init__(self, n_in, n_hidden1, n_hidden2, n_out):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(n_in, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_out)
        )
        
    def forward(self, x):
        out = self.layers(x)
        return out

In [ ]:
# loading the features from memory
def load_features(feature_path):
    listing = os.listdir(feature_path)
    listing = [fname for fname in listing if fname.endswith('.npy')]

    listing_0 = [fname for fname in listing if int(fname.split('_')[-2]) == 0]
    listing_1 = [fname for fname in listing if int(fname.split('_')[-2]) == 1]

    test_split = 0.3
    valid_split = 0.1

    # class 0:
    #print("Splitting class 0.")
    num_0 = len(listing_0)
    random.shuffle(listing_0)
    num_test = int(test_split * num_0)
    num_valid = int(valid_split * num_0)
    num_train = num_0 - num_test - num_valid

    listing_train_0 = listing_0[:num_train]
    listing_valid_0 = listing_0[num_train:num_train+num_valid]
    listing_test_0 =  listing_0[num_train+num_valid:]

    # class 1:
    #print("Splitting class 1.")
    num_1 = len(listing_1)
    random.shuffle(listing_1)
    num_test = int(test_split * num_1)
    num_valid = int(valid_split * num_1)
    num_train = num_1 - num_test - num_valid

    listing_train_1 = listing_1[:num_train]
    listing_valid_1 = listing_1[num_train:num_train+num_valid]
    listing_test_1 =  listing_1[num_train+num_valid:]

    #print("Deleting some files from the listing.")
    mult = 5
    if len(listing_train_0) > mult*len(listing_train_1):
        random.shuffle(listing_train_0)
        listing_train_0 = listing_train_0[:(mult*len(listing_train_1))]

    #print("reorganizing train, validate and test sets.")
    listing_train = listing_train_0 + listing_train_1
    listing_valid = listing_valid_0 + listing_valid_1
    listing_test  = listing_test_0  + listing_test_1
    print(f"Number of samples (train/valid/test): {len(listing_train)}/{len(listing_valid)}/{len(listing_test)}")

    # the actual loading begins...
    dummy = np.load(f"{feature_path}/{listing[0]}")

    #print("Pre-allocating...")
    X_train = np.zeros( (len(listing_train),dummy.shape[0]) )
    Y_train = np.zeros(  len(listing_train) )

    X_valid = np.zeros( (len(listing_valid),dummy.shape[0]) )
    Y_valid = np.zeros(  len(listing_valid) )
    
    X_test =  np.zeros( (len(listing_test),dummy.shape[0]) )
    Y_test =  np.zeros(  len(listing_test) )
   
    
    for X,Y,files in zip([X_train, X_valid, X_test], 
                         [Y_train, Y_valid, Y_test],
                         [listing_train, listing_valid, listing_test]):

        #print("Loading Train, Validate, Test from Memory")
        for i,fname in enumerate(files):
            y = int(fname.split('_')[-2])
            assert(y == 0 or y == 1)
            x = np.load(f"{feature_path}/{fname}")
            
            X[i,:] = x
            Y[i] = y

    
    X_train[X_train == np.inf] = 1
    X_valid[X_valid == np.inf] = 1
    X_test[X_test == np.inf] == 1
    return X_train, Y_train, X_valid, Y_valid, X_test, Y_test


feature_path = f"files/extracted_features_mat/"
X_train, Y_train, X_valid, Y_valid, X_test, Y_test = load_features(feature_path)

print(f'Train size: {len(X_train)}, Validation size: {len(X_valid)}, Test size: {len(X_test)}')


In [ ]:
# show distribution of labels in train and test set
from collections import Counter
train_labels = Y_train
valid_labels = Y_valid
test_labels  = Y_test
print(f'Train label distribution: {Counter(train_labels)}')
print(f'Validation label distribution: {Counter(valid_labels)}')
print(f'Test label distribution: {Counter(test_labels)}')

# from numpy to torch
X_train = torch.tensor(X_train, dtype=torch.float32)
X_valid = torch.tensor(X_valid, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_train = torch.tensor(Y_train, dtype=torch.long)
y_valid = torch.tensor(Y_valid, dtype=torch.long)
y_test  = torch.tensor(Y_test,  dtype=torch.long)

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler((-1,1))
X_train = torch.Tensor(scaler.fit_transform(X_train))
X_valid = torch.Tensor(scaler.transform(X_valid))
X_test =  torch.Tensor(scaler.transform(X_test))

# create Datasets
train_dataset = TensorDataset(X_train, y_train)
valid_dataset = TensorDataset(X_valid, y_valid)
test_dataset  = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
#test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

In [ ]:
# creating the model
model = NeuralNetwork(X_train.shape[1], 64, 128, 2).to(device)
print(model)

In [ ]:
# training (with validation loss)
from torchsummary import summary

input_size = X_train.shape[1]

# show model summary
summary(model, input_size=(input_size,))

num_0 = torch.sum(y_train == 0)
num_1 = torch.sum(y_train == 1)

loss_fn = nn.CrossEntropyLoss(weight=torch.tensor([1/num_0, 1/num_1])) # weights !!! 
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)

epochs = 200
losses = []
validation_losses = []
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0

    for batch_X, batch_Y in train_loader:
        # Forward
        batch_X, batch_Y = batch_X.to(device), batch_Y.to(device)

        outputs = model(batch_X)
        loss = loss_fn(outputs, batch_Y)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    # validation loss:
    model.eval()
    with torch.no_grad():
        valid_outputs = model(X_valid.to(device))
        loss = loss_fn(valid_outputs, y_valid)
    validation_losses.append(loss)
    torch.save(model.state_dict(), f"files/models/ANN_IAML_afterEpoch{epoch}.pth")
    
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss / len(train_loader):.4f}")
    print(f"Validation Loss: {validation_losses[-1]:.4f}")
    losses.append(epoch_loss / len(train_loader))


plt.plot(range(epochs), losses)
plt.plot(range(epochs), validation_losses)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(["training loss", "validation loss"])
plt.title('Training Loss over Epochs')

In [ ]:
# Evaluation on Test set
from sklearn.metrics import confusion_matrix

model.eval()

classes = ['non-nodule', 'nodule']

with torch.no_grad():
    test_outputs = model(X_test)
    _, predicted = torch.max(test_outputs.data, dim=1)
    confmat = confusion_matrix(y_test.numpy(), predicted.numpy())
    
# show confusion matrix
import seaborn as sns
plt.figure(figsize=(8,6))
sns.heatmap(confmat, annot=True, fmt='g', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')

# calculate accuracy from confusion matrix

accuracy = np.diag(confmat).sum() / confmat.sum()
print(f'Accuracy on train set: {accuracy.item()*100:.2f}%')

In [ ]:
# Plotting of saved confusion matrices

#confmat = np.array([[203019, 22607], [86, 379]]) # CNN, undersample & weighted classes
#confmat = np.array([[214016, 11610], [121, 344]]) # CNN, undersampled, no weights

#confmat = np.array([[53758, 1404], [223, 238]]) # ANN, undersampled & weighted classes
confmat = np.array([[55105, 57], [396, 65]])    # ANN, undersampled, no weights

import eval_metrics

eval_metrics.evaluate(confmat)


